# Step 4: Model Evaluation & Performance Review

This notebook evaluates the fine-tuned LLMs by testing them against a random subset of our dataset. It measures their ability to output valid JSON and their exact correctness across multiple CRM fields.

## 0. Setup Google Colab (Mount Drive)
Run this block to mount your Google Drive and enter the project directory.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd "/content/drive/MyDrive/04 - University/04.02 - Adelaide University/04.02.02 - Sem 1 2026/02 - Deep Learning Applications/0 - Final Project/call-contextual-extractor"
except ImportError:
    print("Not running in Google Colab, skipping drive mount.")

## 1. Run Evaluator Pipeline
We will now feed 20 test transcripts into each model (Qwen3.5 2B, Qwen3.5 0.8B, Gemma-4 E4B) and compare their structured JSON responses directly against the Ground Truth created by Gemini.
This script directly tracks:
1. JSON Parsability (can it output valid JSON?)
2. Boolean Accuracy (Busy matching)
3. Exact String Matching (Sector, Scheduled At)
4. Mean Absolute Error (Interested, Rating)

In [ ]:
!python pipeline/evaluator.py --models data/models/Qwen3.5-2B_lora data/models/Qwen3.5-0.8B_lora data/models/gemma-4-E4B_lora

## 2. Visualize Results
Let's load the `evaluation_results.csv` and plot the performance story.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    df = pd.read_csv("data/evaluation_results.csv")
    display(df)
    
    # Set up the plot grid
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")
    
    # 1. JSON Parsability
    sns.barplot(data=df, x="Model", y="Valid_JSON_%", ax=axes[0], palette="Blues_d")
    axes[0].set_title("Valid JSON Output Rate (%)")
    axes[0].set_ylim(0, 100)
    
    # 2. Boolean & String Matches
    df_melted = df.melt(id_vars="Model", value_vars=["Busy_Accuracy_%", "Sector_Match_%", "Schedule_Match_%"], 
                        var_name="Metric", value_name="Score")
    sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", ax=axes[1])
    axes[1].set_title("Field Exact Match Accuracy (%)")
    axes[1].set_ylim(0, 100)
    
    # 3. Mean Absolute Error (MAE) for Ratings
    df_mae = df.melt(id_vars="Model", value_vars=["Interested_MAE", "Rating_MAE"], 
                     var_name="Metric", value_name="MAE")
    sns.barplot(data=df_mae, x="Model", y="MAE", hue="Metric", ax=axes[2])
    axes[2].set_title("Rating Error (MAE - Lower is Better)")
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("evaluation_results.csv not found! Run the evaluator cell first.")